In [1]:
%load_ext autoreload
%autoreload 2

%gui qt

In [2]:
import os
import sys
import importlib
import cv2
import time

gui_source_dir = os.path.expanduser("~/Documents/talmolab/repos/lucid_lite/gui_source")
josh_source_dir = os.path.expanduser("~/Documents/talmolab/repos/lucid_lite")

# add folders to Python's system path so it can find local imports
if gui_source_dir not in sys.path:
    sys.path.insert(0, gui_source_dir)
    sys.path.insert(0, josh_source_dir)

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from tqdm import tqdm
from collections import defaultdict

import numpy as np
import pandas as pd
import sleap_io as sio
from scipy.optimize import linear_sum_assignment

from PySide6.QtGui import QImage

from lucid_lite.gui_source import graph_window
from graph_window import GroupGraphWindow

from lucid_lite.gui_source import main
from lucid_lite.gui_source import colors
from lucid_lite.gui_source.pose_data import Identity  
from lucid_lite.gui_source.colors import next_palette_color     
from lucid_lite.gui_source import labels as L   # labels are sio.Labels now (Labels class removed)

from lucid_lite.josh_source import geometry
from lucid_lite.josh_source import track_push
from lucid_lite.josh_source import tracker
from lucid_lite.josh_source import graphs
import luc3d_tracker_helper as lt

In [3]:
node_weights = {
    'Nose':   0.7,
    'Ear_R':  0.7,
    'Ear_L':  0.7,
    'TTI':    1,
    'TailTip':0,
    'Head':   1,
    'Trunk':  0.8,
    'Tail_0': 0,
    'Tail_1': 0,
    'Tail_2': 0,
    'Shoulder_left':  0.7,
    'Shoulder_right': 0.7,
    'Haunch_left':    0.7,
    'Haunch_right':   0.7,
    'Neck': 0.7
}

SESSION = "/Users/joshuapark/Documents/talmolab/lucid_folders/10072022145420"

# labels is now a multi-video sio.Labels (one Video per camera); calibration
# still comes from SESSION. Swap h5_dir to predictions_h5s for raw SLEAP.
labels = L.from_aggregated_h5(
    SESSION,
    # h5_dir="/Users/joshuapark/Documents/talmolab/lucid_folders/sleap_nn_predictions_h5s",
    h5_dir="/Users/joshuapark/Documents/talmolab/lucid_folders/predictions_h5s",
    session_idx=70,
)                                              # -> sleap_io.Labels
poses = L.to_arrays(SESSION, labels)           # {cam: (n_frames, n_tracks, n_nodes, 2)}
app, window = main.main(SESSION, labels=labels)

objc[49899]: Class AVFFrameReceiver is implemented in both /Users/joshuapark/Documents/talmolab/repos/lucid_lite/.venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x123a743a8) and /Users/joshuapark/Documents/talmolab/repos/lucid_lite/.venv/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x133e383a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[49899]: Class AVFAudioReceiver is implemented in both /Users/joshuapark/Documents/talmolab/repos/lucid_lite/.venv/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x123a743f8) and /Users/joshuapark/Documents/talmolab/repos/lucid_lite/.venv/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x133e383f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [15]:
node_visible = ~np.isnan(poses['back']).all(axis=(-1, -2))
node_visible = node_visible.mean(axis=0)
node_visible



array([0.57408929, 0.38082717, 0.37551356, 0.18646946])

In [18]:
window.session

<pose_data.Session(0xc50340c30) at 0x312e73b40>

In [50]:
importlib.reload(tracker)

mft = tracker.MultiFrameTrack(
    window=window,
    node_weights=node_weights,
    start=0,
    end=3217,
)
mft.track(verbose=True)

mft.push_assignments_to_gui()

950
1133
1134
1209
1210
1211
1212
1213
1214
1215
1216
1217
1218
1219
3204
3208
3209
3210
3211
3212
3213
3214
3215
3216


100%|██████████| 3217/3217 [00:00<00:00, 12391.10it/s]


Num invalid instances                   0
Num nonmatch instances               4396
Num nonmatch groups                     0
Num nonmatch group instances            0

Total Instances                      5406
Fraction Valid                    1.00000
Fraction Matched Single           0.18683
Fraction Matched With Group       0.18683


In [47]:
window.session.max_frame

18254

In [46]:
poses['back'].shape

(18255, 4, 15, 2)

In [ ]:
len(window.session.frame_group(i).instances.values())


1